# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. The dataset follows the Croissant schema for AI/ML-ready metadata and is a practical demonstration of standardized, FAIR-compliant health data from Africa.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
meta = dataset.metadata  # This is an mlcroissant.Metadata object
print(f"{meta.name}: {meta.description}\n")
print(f"Published: {meta.date_published} | Version: {meta.version}")
print(f"Citation: {meta.cite_as}\n")

## 2. Data Overview
Review available record sets, their `@id`, and contained fields/columns. All references use `@id` to guarantee schema consistency.

In [ ]:
# List all available record sets and their fields via their @id
rs_list = [r for r in meta.record_sets]
print(f"Found {len(rs_list)} record sets.\n")
for rs in rs_list:
    print(f"RecordSet: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for f in rs.fields:
        print(f"    - {f.id}: {f.name} (dataType: {f.data_type})")
    print(f"  Columns (if any):")
    for c in rs.columns:
        print(f"    - {c.id}: {c.name} (dataType: {c.data_type})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame. Use the record set and field/column `@id` from the overview above.

In [ ]:
# Collect all record_set IDs
record_set_ids = [rs.id for rs in meta.record_sets]

# For demonstration, let's extract the first record set (usually the main survey data)
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}\n  Sample rows:")
    display(df.head(2))
    print("---\n")
# For further analysis, let's work with the first record set
if len(record_set_ids) == 0:
    raise Exception("No record sets found in this dataset.")
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]
print(f"Selected main record set: {main_record_set_id}")
print(f"Columns: {main_df.columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filter by survey scores, normalize, group by demographic fields, etc. Reference all fields by their `@id`.

In [ ]:
# Example: Analyze GAD-7 score
# Find the field for GAD-7 score (by @id or best match by column name)
# Replace below with the actual @id of the GAD-7 field if known
gad7_field_id = None
for rs in meta.record_sets:
    for f in rs.fields:
        if f.name.lower().startswith("gad") and "score" in f.name.lower():
            gad7_field_id = f.id
            print(f"Detected GAD-7 field: {gad7_field_id} -> column: {f.name}")
            break
    if gad7_field_id:
        break
# Fallback to possible column names
if gad7_field_id is None:
    # Try column match
    for c in main_df.columns:
        if "gad" in c.lower() and "score" in c.lower():
            gad7_field_id = c
            print(f"Guessed GAD-7 score column: {gad7_field_id}")
            break

if gad7_field_id is None:
    raise Exception("GAD-7 score field not found by name. Please check metadata.")

# Clean column name if prefixed by @id
gad7_field = gad7_field_id
df = main_df.copy()

# Show distribution
print("GAD-7 score summary:")
print(df[gad7_field].describe())

# Remove outliers (e.g., lower bound >= 0, upper <= 21 for GAD-7)
filtered_df = df[(df[gad7_field] >= 0) & (df[gad7_field] <= 21)]
print(f"\nRecords with valid GAD-7 scores: {len(filtered_df)} out of {len(df)}")

# Normalize GAD-7
filtered_df[f"{gad7_field}_normalized"] = (
    filtered_df[gad7_field] - filtered_df[gad7_field].mean()
) / filtered_df[gad7_field].std()
print(filtered_df[[gad7_field, f"{gad7_field}_normalized"]].head())

# Group by a demographic field (e.g., gender, education)
# Try to find @id for gender or education
group_field_id = None
candidates = [c for c in main_df.columns if 'gender' in c.lower() or 'education' in c.lower()]
if candidates:
    group_field_id = candidates[0]
    print(f"Grouping by: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[gad7_field].mean().reset_index()
    print(grouped_df)
else:
    print("No demographic grouping field (e.g., gender, education) found for grouping.")

## 5. Visualization
Visualize the GAD-7 score distribution and (if available) comparison by gender or education group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 scores
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[gad7_field].dropna(), bins=range(0,23), kde=True)
plt.title("Distribution of GAD-7 Scores")
plt.xlabel("GAD-7 Score")
plt.ylabel("Count")
plt.show()

# Boxplot by group (if available)
if group_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[gad7_field])
    plt.title(f'GAD-7 Score Distribution by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel('GAD-7 Score')
    plt.show()

## 6. Conclusion
We successfully loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the Croissant schema and `mlcroissant`. We demonstrated how to:
- Discover record sets and field structure (using `@id`)
- Extract survey records into Pandas DataFrames
- Perform basic EDA: filtering, normalization, and grouping by a demographic field
- Visualize GAD-7 mental health scores and compare across groups

The data enables studies of mental health patterns with robust, well-described metadata, illustrating reproducible, FAIR principles for real-world health datasets in Africa.